# MS in CS Thesis - Generating Synthetic Data

Name - Shreeya Sharda

Course - CS6999

**The research objectives of this file includes the following:**


*   Implement zero shot, one-shot, few-shot (for minority classes)





**Installing Ollama**

In [ ]:
%load_ext cudf.pandas
#This is a magic Python command (Source: On the 3rd icon in the left tab of google collab, this command is listed. We leveraged Google Collab in the thesis)
#This promotes improved runtime in the pandas python library
import pandas as pd
from google.colab import drive
import os, subprocess, time, signal
import re

# Mount Google Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install ollama
import ollama

**Core lines for installing Ollama**

In [ ]:
import time

subprocess.run("sudo apt-get update && sudo apt-get install -y zstd", shell=True, check=True)
# We need to install zstd b/c in the previous method we tested, collab gave an error saying it was not installed
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)


# Source: https://pinggy.io/blog/running_ollama_on_google_colab_with_pinggy/
proc = subprocess.Popen(["ollama", "serve"])
print("Ollama server launched")

time.sleep(10)

!ollama pull llama3:8b-instruct-q4_0

Ollama server launched



**We are using the instruct fine-tuned Llama3.1 library for text generation**

In [ ]:
!ollama list

NAME                       ID              SIZE      MODIFIED               
llama3:8b-instruct-q4_0    365c0bd3c000    4.7 GB    Less than a second ago    


In [ ]:
MODEL_NAME = 'llama3:8b-instruct-q4_0'
MIN_WORDS = 25
MAX_WORDS = 200
TOKENS_MAX = 280


**System Level Prompt Template - Zero Shot**

In [ ]:
zero_system_prompt = f'''Act as a tourist visiting Juneau, Alaska that is writing a review about their experience. Output only the review text.

Act as a synthetic data generator for online tourism reviews for Juneau, Alaska. Output only the review text.

Rules:
1. Do not include any extra info, such as a title, review length, rating, symbols, notes, explanations, or conversational phrases like 'Here is the review', 'NOTE:', 'As an AI', etc.
2. The review should mimic the style of TripAdvisor reviews.
3. The review text should be between {MIN_WORDS} and {MAX_WORDS} words long.
4. The sentiment of the review must correspond to the specified rating: 1 = terrible, 2 = poor, 3 = average.
5. The defintions of the 2 sentiments are as follows:
    Negative - Signifies intense negative emotion; indicates dissatisfaction.
    Neutral - The tourist experienced a mix of good and bad points. Overall experience was not the best, however it was not negative


'''

**System Level Prompt Template - One Shot**

In [ ]:
one_system_prompt = f'''


Act as a synthetic data generator for online tourism reviews for Juneau, Alaska. Output only the review text.

Rules:
1. Do not include any extra info, such as a title, review length, rating, symbols, notes, explanations, or conversational phrases like 'Here is the review', 'NOTE:', 'As an AI', etc.
2. The review should mimic the style of TripAdvisor reviews.
3. The review text should be between {MIN_WORDS} and {MAX_WORDS} words long.
4. The sentiment of the review must correspond to the specified rating: 1 = terrible, 2 = poor, 3 = average.
5. The defintions of the 2 sentiments are as follows:
    Negative - Signifies intense negative emotion; indicates dissatisfaction.
    Neutral - The tourist experienced a mix of good and bad points. Overall experience was not the best, however it was not negative

Here are some examples of reviews associated with ratings. Maintain high lingustic diversity in the reviews. Do not copy the language of the example reviews.

Review: There is absolutely No Content up there! $45 for what?! So they can employ a few folks? The “Award Winning” video in the “theatre” when it works, is nothing more than a ridiculously novice attempt at giving a 30 minute history you could read about in 2 minutes online. The expected view was “meh.” The food was the worst I had in my six meals in town. No! The worst I’ve had in weeks— including airport meals! The raptor exhibit was closed. The coffee shop with outdoor seating was closed. There is, however, yet another huge waste of time and money gift shop offering all the same stuff as in town— sans the $45 ride. Stay away! Save your money.
Sentiment: Negative

Review: I was expecting a lot more from the Mendenhall Glacier excursion I joined through Holland America. GREATLY disappointed by the Mendenhall Glacier. I found it utterly underwhelming. You basically go and get a picture of the glacier, and then go up to the presentation centre for a brief 15 min movie to kill some time before the bus comes back to pick you up."
Sentiment: Negative

Review: I give this a 3 not because of the glacier, but because of the huge crowd of tourists, some not behaving very well for which the glacier is not to blame. The park service did their best. No-one looking at the glacier, the lake and run off can possible doubt global warming,
Mallele.
Sentiment: Neutral

'''




**System Level Prompt Template - Few Shot**

In [ ]:
few_system_prompt = f'''

Act as a synthetic data generator for online tourism reviews for Juneau, Alaska. Output only the review text.

Rules:
1. Do not include any extra info, such as a title, review length, rating, symbols, notes, explanations, or conversational phrases like 'Here is the review', 'NOTE:', 'As an AI', etc.
2. The review should mimic the style of TripAdvisor reviews.
3. The review text should be between {MIN_WORDS} and {MAX_WORDS} words long.
4. The sentiment of the review must correspond to the specified rating: 1 = terrible, 2 = poor, 3 = average.
5. The defintions of the 2 sentiments are as follows:
    Negative - Signifies intense negative emotion; indicates dissatisfaction.
    Neutral - The tourist experienced a mix of good and bad points. Overall experience was not the best, however it was not negative

Here are some examples of reviews associated with ratings. Maintain high lingustic diversity in the reviews. Do not copy the language of the example reviews.

Review: There is absolutely No Content up there! $45 for what?! So they can employ a few folks? The “Award Winning” video in the “theatre” when it works, is nothing more than a ridiculously novice attempt at giving a 30 minute history you could read about in 2 minutes online. The expected view was “meh.” The food was the worst I had in my six meals in town. No! The worst I’ve had in weeks— including airport meals! The raptor exhibit was closed. The coffee shop with outdoor seating was closed. There is, however, yet another huge waste of time and money gift shop offering all the same stuff as in town— sans the $45 ride. Stay away! Save your money.
Sentiment: Negative

Review: I was expecting a lot more from the Mendenhall Glacier excursion I joined through Holland America. GREATLY disappointed by the Mendenhall Glacier. I found it utterly underwhelming. You basically go and get a picture of the glacier, and then go up to the presentation centre for a brief 15 min movie to kill some time before the bus comes back to pick you up."
Sentiment: Negative

Review: We arrived here from a cruise ship, on May 6 2016, interested in the history of Juneau, and what the place represented.
Reading some other reviews on this place, we did not get herded into any one particular area, instead, entered through the souvenir shop and sat at a table with friends, and, after our initial order, were left alone to enjoy the atmosphere.
Another group of elderly travellers on our tour were seated next to us. The afternoon started ok, however, being new to the area, tourists do like to visit other places within the town.
As the elderly travellers finished their drinks and meal, then began to leave, the country & western male entertainer said, "Must be feed in' time on the boat..! How rude are you lot, walking out on me during the middle of my show..!"
On hearing this, Our group decided it was time to go, so, after numerous attempts to attract the attention of the barmaid, I decided to go to the cash register area to finalise our account.
After ten minutes of waiting, and requesting another attendant sort our bill out, the original barmaid, a short blonde haired woman in her 40's with very bad attitude, approached and said, "It's my birthday, and you're ruining my 15 minutes of fame..! Wait until the song is finished...!"
Needless to say the tip we were about to offer, was not given.
If this place thrives on the tourist dollar, with attitudes of the staff like we encountered, this place doesn't have much of a future. This place is fortunate cruises visit. If you ran a place in Australia with attitude like this, you would not survive too long.
Mallele.
Sentiment: Neutral

Review: When I booked our Alaska cruise the only thing my husband cared about was going Kayaking so I booked this excursion through Costco. As we wandered through the town I turned my phone on and got a text that the excursion was cancelled "due to weather". I found this startling given the information provided:
Operates in all weather conditions, please dress appropriately. Subject to favorable weather conditions. If canceled due to poor weather, you will be given the option of an alternative date or full refund.
I was further disturbed upon learning in town that the roads had been flooded for weeks, so clearly this tour was not an option long before we received our text message. We were left to scramble to find another excursion when this company could have provided the information far sooner and we would have a much better experience in Juneau.
Furthermore, despite the text that a full refund would be given, I had to call the company and was told, rudely, that i needed to call you know the other people which eventually was determined to be the travel agent/Costco. I called Costco who had to call the company to confirm the cancelled trip before I could get my refund. I found the entire process highly unacceptable.
Sentiment: Negative

Review: Having to spend one night in Juneau after a cruise, I booked a room at the Prospector. The price seemed kind of high for what I expected we'd have, but the room was advertised as "water view suite with a King Bed".
In was in fact a suite - rather large actually (I could have easily slept two people on the floor of the walk in closet). However the King Bed - the primary reason I got the room - was a Queen, and not a particularly comfortable one at that - and the "water view" was of an old dock and a waterfront warehouse right across the street from the hotel, which was situated on the main drag in Juneau. So we got to listen to heavy traffic all night long. Not really the peaceful water experience I had anticipated.
If there was a plus to the place, it was that it's walking distance from the main downtown area - and right next door to a nice museum.
Sentiment: Negative

Review: All I wanted was to take a warm relaxing bath when I got there but arrived to find a dirty scum ring in the tub. Eeeeew.
I called the front desk to ask if they would please bring cleaning supplies to my room to clean it out.
The young fellow who answered the phone had a strange (uncomfortable) way of replying- he repeated each thing I said as a question. "Your bathtub is dirty? Your bathtub has an oily ring around the inside?"
after that he was very nice and came to the room right away and went into the bathroom with cleaner and a rag.
I said thank you, he left and I went to enjoy my bath but found the tub still had the same scum line and now the addition of a few curly hairs.
Ugh. I just used my wash rag and shampoo and scrubbed it out myself.
The people are very cordial here and no other problems whatsoever but Ay! Cleaning skills are a zero.😁😩😶 On the phone they did offer to switch rooms if we wanted , it just didn't seem necessary if they could clean the tub out. I was tired and the rest of the room was fine.
Sentiment: Neutral


'''

**Define the variables necessary for the user level prompt template**

In [ ]:
locations_text = """
Mendenhall Glacier Tour
UnCruise Adventures Cruise
Alaska Travel Adventures Tourist Information Center
Mendenhall Lake Canoe Adventure Tour
3.5 Hour Whale Watching Tour
Mendenhall Glacier Trolley Tour
Mendenhall Glacier Sea Kayaking Tour
Mendenhall Lake Kayak Tour
Taku Lodge Feast & 5-Glacier Seaplane Discovery Tour
Juneau Underground Gold Mine and Panning Experience Tour
Paddle with Whales Kayak Adventure Tour
Best Western Grandma’s Feather Bed Hotel
Sand Bar Grill Restaurant
Donna’s Restaurant
McGivney’s Sports Bar Restaurant
NorthStar Helicopters Tour
Bear Creek Outfitters Park
Juneau Mercantile & Armory Tour
Travelodge by Wyndham Hotel
Alaska Tales Tour
Mendenhall Glacier Visitor Center
Tracy Arm Fjord Glacier Exploration Tour
Nugget Falls Waterfall Tourist Site
Juneau Lighthouse Tour
Alaska State Museum
Bowl of Pho Restaurant
Valley Restaurant
McDonalds Restaurant
Asiana Gardens Restaurant
Best Western Country Lane Inn
Ramada by Wyndham Juneau Hotel
Travelodge by Wyndham Juneau Hotel
Juneau Shore Excursion: Helicopter Tour and Guided Icefield Walk Tour
Mendenhall Glacier Float Trip Tour
Mendenhall Glacier Visitor Center and Ultimate Whale Watch Combo Tour
Ultimate Juneau Whale Watch and Mendenhall Glacier View Tour
Juneau Wildlife Whale Watching Tour
Whale Watching Adventure in Juneau Tour
Juneau Gold Creek Salmon Bake Tour
Alaska Native Tour and Tram Combo Tour
Juneau Shore Fishing for Alaskan Salmon Tour
Juneau Aspen Suites Hotel
Juneau Hotel
The Rookery Cafe
Silverbow Inn
Narrated Whale Watching Cruise with Glacier Viewpoint Tour
Pearson’s Pond Luxury Suites and Vacation Rentals Hotel
Beachside Villa Luxury Inn
Super 8 Hotel By Wyndham Juneau
Red Dog Saloon Restaurant
Four Points Hotel by Sheraton Juneau
Tracy’s King Crab Shack
The Hangar on the Wharf
"""

locations_list = [f"{loc.strip()}" for loc in locations_text.split('\n') if loc.strip()]


In [ ]:
synthetic_reviews = []

personnas = ["chatty", "sarcastic", "angry", "enthusiastic", "informative", "happy", "sad", "rude", "playful", "old fashioned", "budget-friendly"]

travel_type = ["family", "solo", "business", "couple", "friends"]

aspects = ["cleanliness", "COVID-19", "crowds", "ticket prices", "cruise ships", "staff professionalism", "global warming", "hiking trail", "weather conditions", "food", "wait times", "local residents"]

ratings = [1, 2, 3]

informal_language = ["small typos", "sarcasm", "metaphors", "colloquialism", "emojis"]

**Function to convert synthetic data dataframe to a CSV file**

In [ ]:
def convert_to_csv(synthetic_reviews, file_path):


    df_reviews = pd.DataFrame(synthetic_reviews)

    print(f"\nThe following number of reviews have been generated: {len(df_reviews)}")

    # Convert the dataframe to a csv file
    df_reviews.to_csv(file_path, index=False)




**Installing Anthropic**

Source: https://pypi.org/project/anthropic/


In [ ]:
#pip install anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 8.3 MB/s eta 0:00:00


In [ ]:
# API_KEY = "" #We removed this key from Collab for the purposes of privacy. 

# import os
# from anthropic import Anthropic

# client = Anthropic
# (
#     api_key=API_KEY,
# )

**Generating Data using Anthropic**

In [ ]:
from anthropic.types import ToolTextEditor20250124Param
import random

# We used the "Gemini" feature on Google Collab to debug this code cell (for claude)

def generate_synthetic_data_claude (NUM_INSTANCES, user_rating, personnas, locations, travel_type, informal_language, aspects):

  file_path = f'/content/drive/MyDrive/synthetic_reviews_claude_new_2.csv'


  for i in range(NUM_INSTANCES):

    styles = random.choice(personnas)
    traveler_type = random.choice(travel_type)
    location_visited = random.choice(locations)
    language = random.choice(informal_language)
    aspects = random.choice(aspects)

    user_prompt = f'''Act as a tourist visiting to Juneau, Alaska. Write a review about your experience visiting a location in Juneau, using the following details:

    The type of visit should be: {traveler_type}.
    You are traveling to the following location: {location_visited}.
    The tone of the review should be: {styles}.
    The sentiment of the review should match the following rating: {user_rating}.
    Include 1 case of the following language feature: {language}
    Discuss your sentiment (positive, negative, neutral) towards the following aspect: {aspects}


    Mantain high linguistic diversity in the reviews.'''


    message = client.messages.create (
            model="claude-opus-4-6",
            max_tokens= TOKENS_MAX,
            system=few_system_prompt,
            messages=[
                {'role': 'user', 'content': user_prompt}

            ],
            temperature = 0.7
        )
    review_text = message.content[0].text.strip()


    words = re.findall(r'\b\w+\b', review_text)
    word_count = len(words)

    if MIN_WORDS <= word_count <= MAX_WORDS:
          synthetic_reviews.append({'rating': user_rating, 'review': review_text, 'word_count': word_count})
    else:
          print(f"Length outside of range: {word_count}")

          message_refine = client.messages.create (
                model= "claude-opus-4-6",
                max_tokens= TOKENS_MAX,
                system=f'''Act as a helpful assistant.
                    Your task is to ONLY adjust the review length to be in the word count range {MIN_WORDS} and {MAX_WORDS}.
                    Maintain the linguistic diversity and writing style of the input review.''',
                messages=[
                    {'role': 'user', 'content': review_text}
                ],
                temperature = 0.7
            )
          refined_text = message_refine.content[0].text.strip()

            # MAKE SURE WORD COUNT IS IN RIGHT RANGE
          words_new = re.findall(r'\b\w+\b', refined_text)
          word_count_refined = len(words_new)

          if MIN_WORDS <= word_count_refined <= MAX_WORDS:
              synthetic_reviews.append({'rating': user_rating, 'review': refined_text, 'word_count': len(refined_text.split())  })
          else:
              synthetic_reviews.append({'rating': user_rating, 'review': refined_text, 'word_count': word_count_refined, 'note': "word count outside of range"})
              print("Synthetic review - outside of word range")


    convert_to_csv(synthetic_reviews, file_path)

**Generating Data using Ollama**

In [ ]:
import random

def generate_synthetic_data (NUM_INSTANCES, user_rating, personnas, locations, travel_type, informal_language, aspects):

  file_path = f'/content/drive/MyDrive/synthetic_reviews_300.csv'

  for i in range(NUM_INSTANCES):

    styles = random.choice(personnas)
    traveler_type = random.choice(travel_type)
    location_visited = random.choice(locations)
    language = random.choice(informal_language)
    aspect = random.choice(aspects)

    user_prompt = f'''Act as a tourist visiting to Juneau, Alaska. Write a review about your experience visiting a location in Juneau, using the following details:

    The type of visit should be: {traveler_type}.
    You are traveling to the following location: {location_visited}.
    The tone of the review should be: {styles}.
    The sentiment of the review should match the following rating: {user_rating}.
    Include 1 case of the following language feature: {language}
    Discuss your sentiment (positive, negative, neutral) towards the following aspect: {aspect}


    Mantain high linguistic diversity in the reviews.'''


    response = ollama.chat (
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': few_system_prompt},
                {'role': 'user', 'content': user_prompt}

            ],
            options = {
                "num_predict": TOKENS_MAX,

                "temperature": 1.12

                # "mirostat": 2,
                # "mirostat_tau": 9,
                # "mirostat_eta": 0.6


            },
        )
    review_text = response['message']['content'].strip()


    words = re.findall(r'\b\w+\b', review_text)
    word_count = len(words)

    if MIN_WORDS <= word_count <= MAX_WORDS:
          synthetic_reviews.append({'rating': user_rating, 'review': review_text, 'word_count': word_count})
    else:
          print(f"Length outside of range: {word_count}")

          response_refine = ollama.chat (
                model=MODEL_NAME,
                messages=[
                    {'role': 'system', 'content': f'''Act as a helpful assistant.
                    Your task is to ONLY adjust the review length to be in the word count range {MIN_WORDS} and {MAX_WORDS}: {review_text}.
                    Maintain the linguistic diversity and writing style of the input review.'''}


                ],
                options =
                {
                    "num_predict": TOKENS_MAX,
                    "temperature": 1.12
                    #"mirostat": 2,

                }
            )
          refined_text = response_refine['message']['content'].strip()

            # Re-check that the word count is in the proper range
          words_new = re.findall(r'\b\w+\b', refined_text)
          word_count_refined = len(words_new)

          if MIN_WORDS <= word_count_refined <= MAX_WORDS:
              synthetic_reviews.append({'rating': user_rating, 'review': refined_text, 'word_count': len(refined_text.split())  })
          else:
              synthetic_reviews.append({'rating': user_rating, 'review': refined_text, 'word_count': word_count_refined, 'note': "word count outside of range"})
              print("Synthetic review - outside of word range")

    convert_to_csv(synthetic_reviews, file_path)

**Generate synthetic reviews**

In [ ]:
generate_synthetic_data(300, ratings[2], personnas, locations_list, travel_type, informal_language, aspects)

Length outside of range: 226
Synthetic review - outside of word range

The following number of reviews have been generated: 1
Length outside of range: 230

The following number of reviews have been generated: 2
Length outside of range: 224

The following number of reviews have been generated: 3
Length outside of range: 234

The following number of reviews have been generated: 4

The following number of reviews have been generated: 5
Length outside of range: 219

The following number of reviews have been generated: 6
Length outside of range: 209

The following number of reviews have been generated: 7
Length outside of range: 230
Synthetic review - outside of word range

The following number of reviews have been generated: 8
Length outside of range: 233

The following number of reviews have been generated: 9
Length outside of range: 225

The following number of reviews have been generated: 10
Length outside of range: 218

The following number of reviews have been generated: 11

The follo